# Agent for calling SQL queries


**Chinok Database link:**\
https://github.com/lerocha/chinook-database\
https://docs.yugabyte.com/stable/sample-data/chinook/

In [1]:
# Use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from langchain.agents import AgentType
from langchain_openai import ChatOpenAI

from langchain_community.utilities.sql_database import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini")

## Use SQL Database

In [ ]:
# # Download sql database
# https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Mauh_UvY4eK2SkcHj_b8Tw/chinook-mysql.sql
# ANOTHER LINK: https://github.com/lerocha/chinook-database/blob/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql

--2026-02-15 12:15:19--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/Mauh_UvY4eK2SkcHj_b8Tw/chinook-mysql.sql
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... failed: Temporary failure in name resolution.
wget: unable to resolve host address ‘cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud’


In [ ]:
# mysql_username = 'root'  # Replace with your server connect information
# mysql_password = 'TGbWkWxf2iPljYE3mYm8ilBj' # Replace with your server connect information
# mysql_host = '172.21.155.85' # Replace with your server connect information
# mysql_port = '3306' # Replace with your server connect information
# database_name = 'Chinook'
# mysql_uri = f'mysql+mysqlconnector://{mysql_username}:{mysql_password}@{mysql_host}:{mysql_port}/{database_name}'
# db = SQLDatabase.from_uri(mysql_uri)

## Use Sqlite Database

In [ ]:
# # Download sqlite database
# https://github.com/lerocha/chinook-database/blob/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite

In [4]:
# Load database
db = SQLDatabase.from_uri("sqlite:///Chinook_Sqlite.sqlite")

In [7]:
agent_executor = create_sql_agent(llm=llm, db=db, verbose=True, handle_parsing_errors=True, agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION)

In [ ]:
agent_executor.invoke(
    "How many Album are there in the database?"
)



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables  
Action Input: ""  Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, TrackThe relevant table for albums is the "Album" table. I will check the schema of the Album table to understand its structure before querying for the count of albums.  
Action: sql_db_schema  
Action Input: Album  
CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/To find out how many albums are in the database, I will construct a query to count the number of rows in the "Album" table. 

Action: sql_db_query_checker  
Action Input: "SELECT COUNT(*) as AlbumCount FROM Album;"  ```sql
SELECT COUNT(*) as Albu

{'input': 'How many Album are there in the database?',
 'output': 'There are 347 albums in the database.'}

In [ ]:
agent_executor.invoke(
    "Describe the PlaylistTrack table"
)



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables  
Action Input: ""  Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, TrackI will check the schema of the PlaylistTrack table to understand its structure and the relevant columns it contains.  
Action: sql_db_schema  
Action Input: "PlaylistTrack"  
CREATE TABLE "PlaylistTrack" (
	"PlaylistId" INTEGER NOT NULL, 
	"TrackId" INTEGER NOT NULL, 
	PRIMARY KEY ("PlaylistId", "TrackId"), 
	FOREIGN KEY("TrackId") REFERENCES "Track" ("TrackId"), 
	FOREIGN KEY("PlaylistId") REFERENCES "Playlist" ("PlaylistId")
)

/*
3 rows from PlaylistTrack table:
PlaylistId	TrackId
1	3402
1	3389
1	3390
*/I now know the final answer.  
Final Answer: The PlaylistTrack table has the following structure: it contains two columns, PlaylistId and TrackId, both of which are integers. PlaylistId is a foreign key referencing the Playlist table, and TrackId is a foreign key referencing the Track table

{'input': 'Describe the PlaylistTrack table',
 'output': 'The PlaylistTrack table has the following structure: it contains two columns, PlaylistId and TrackId, both of which are integers. PlaylistId is a foreign key referencing the Playlist table, and TrackId is a foreign key referencing the Track table. The combination of PlaylistId and TrackId forms the primary key of the table.'}

In [9]:
agent_executor.invoke(
    "Can you left join table Artist and table Album by ArtistId? Please show me 5 Name and AlbumId in the joint table."
)



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables  
Action Input: ""  Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, TrackI have identified the tables in the database: Album and Artist. Next, I will check the schema of these two tables to find the relevant columns for the left join operation.

Action: sql_db_schema  
Action Input: "Album, Artist"  
CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/To perform a left join between the Artist an

{'input': 'Can you left join table Artist and table Album by ArtistId? Please show me 5 Name and AlbumId in the joint table.',
 'output': 'The joint table shows the following results: \n- AC/DC (AlbumId: 1)\n- AC/DC (AlbumId: 4)\n- Accept (AlbumId: 2)\n- Accept (AlbumId: 3)\n- Aerosmith (AlbumId: 5)'}